# 9. El Deslizador: Análisis de Trayectorias Anticipatorias

**Hipótesis (H3 de la tesis):** la Fase 2 (S2) no es estática. Su representación vectorial cambia gradualmente **antes** de la transición hacia REM o SWS, demostrando *memoria latente* sobre el destino.

## Diseño del experimento
Se construyen 7 ventanas sintéticas de tamaño N=6 que migran token a token desde S2 puro hacia el destino:

| Paso | S2 → REM | S2 → SWS |
|:---:|:---:|:---:|
| 0 | (2,2,2,2,2,2) | (2,2,2,2,2,2) |
| 1 | (2,2,2,2,2,4) | (2,2,2,2,2,3) |
| … | … | … |
| 6 | (4,4,4,4,4,4) | (3,3,3,3,3,3) |

Para cada paso se mide:
1. **Similitud coseno con el origen** s(v_t, v_0) → mide cuánto se aleja del estado S2 puro.
2. **Similitud coseno paso a paso** s(v_t, v_{t-1}) → detecta el "momento de decisión" (valle).

**Control (surrogate):** se entrena un W2V sobre un corpus permutado aleatoriamente (mismo vocab, orden temporal destruido). Si la separación observada es real, debe colapsar bajo el control.

> **Decisión metodológica:** para el deslizador se entrena un Word2Vec propio con **min_freq=1** (en vez del **min_freq=2** del notebook 8), de modo que los 6-gramas intermedios —raros en el sueño real— entren al vocabulario. El resto de hiperparámetros (dim=32, ventana=1, stride=1, 30 épocas) es idéntico al notebook 8; la auditoría confirma cobertura 14/14 de los tokens del deslizador.

## 1. Imports y configuración

In [1]:
import os
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

RUTA_BASE = Path('..').resolve()
RUTA_DATOS = RUTA_BASE / 'dataset' / 'epocas_unificado.csv'
RUTA_MODELOS = RUTA_BASE / 'modelos'
RUTA_IMG = RUTA_BASE / 'imagenes' / 'deslizador'
RUTA_STATS = RUTA_BASE / 'estadisticas'
RUTA_IMG.mkdir(parents=True, exist_ok=True)

# Estilo
plt.style.use('default')
pio.templates.default = 'plotly_white'

COLOR_REM = '#9B59B6'      # púrpura
COLOR_SWS = '#3498DB'      # azul
COLOR_SURR_REM = '#C8B6D4' # púrpura claro punteado
COLOR_SURR_SWS = '#A6C9E2' # azul claro punteado
COLOR_GRIS = '#7F8C8D'

SEMILLA = 42
torch.manual_seed(SEMILLA)
np.random.seed(SEMILLA)
DISPOSITIVO = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DISPOSITIVO}')

Dispositivo: cuda


## 2. Carga del corpus y modelo W2V (nb8)

In [2]:
df = pd.read_csv(RUTA_DATOS)
df = df.sort_values(['paciente', 'epoca']).reset_index(drop=True)
print(f'Épocas: {len(df):,} | Pacientes: {df["paciente"].nunique()}')

N = 6  # tamaño de n-grama (decisión nb7)
def token_a_str(tup):
    return '-'.join(str(x) for x in tup)
def str_a_tup(s):
    return tuple(int(x) for x in s.split('-'))

# Construir corpus de 6-gramas con stride=1 (idéntico a nb8)
corpus = []
for pid, g in df.groupby('paciente', sort=False):
    seq = g['fase_num'].astype(int).tolist()
    for i in range(len(seq) - N + 1):
        corpus.append(token_a_str(seq[i:i+N])) 
print(f'Tokens (6-gramas) en corpus: {len(corpus):,}')

Épocas: 112,809 | Pacientes: 127
Tokens (6-gramas) en corpus: 112,174


In [3]:
class ModeloWord2Vec(nn.Module):
    def __init__(self, tam_vocab, dim_embedding):
        super().__init__()
        self.embeddings = nn.Embedding(tam_vocab, dim_embedding)
        self.salida = nn.Linear(dim_embedding, tam_vocab)
    def forward(self, x):
        return self.salida(self.embeddings(x))

class DatasetSkipGram(Dataset):
    def __init__(self, indices, ventana):
        self.pares = []
        for i in range(len(indices)):
            for j in range(max(0, i-ventana), min(len(indices), i+ventana+1)):
                if i != j:
                    self.pares.append((indices[i], indices[j]))
    def __len__(self): return len(self.pares)
    def __getitem__(self, i):
        a, b = self.pares[i]
        return torch.tensor(a), torch.tensor(b)

def entrenar_w2v(corpus_tokens, min_freq, dim, epocas, lr, ventana, batch, dispositivo):
    freq = Counter(corpus_tokens)
    vocab = sorted([t for t, c in freq.items() if c >= min_freq])
    idx = {t: i for i, t in enumerate(vocab)}
    indices = [idx[t] for t in corpus_tokens if t in idx]
    print(f'  vocab (min_freq={min_freq}): {len(vocab):,}')
    print(f'  pares de entrenamiento: aprox {2*len(indices)*ventana:,}')
    ds = DatasetSkipGram(indices, ventana=ventana)
    loader = DataLoader(ds, batch_size=batch, shuffle=True, num_workers=0)
    modelo = ModeloWord2Vec(len(vocab), dim).to(dispositivo)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(modelo.parameters(), lr=lr)
    hist = []
    for ep in range(1, epocas + 1):
        modelo.train()
        total = 0; nb_ = 0
        for a, b in loader:
            a, b = a.to(dispositivo), b.to(dispositivo)
            opt.zero_grad()
            loss = crit(modelo(a), b)
            loss.backward(); opt.step()
            total += loss.item(); nb_ += 1
        media = total / max(nb_, 1)
        hist.append(media)
        if ep == 1 or ep % 5 == 0 or ep == epocas:
            print(f'  época {ep:3d}/{epocas}  pérdida={media:.4f}')
    return modelo, idx, hist

# Parámetros (min_freq=1 a petición del usuario para análisis del deslizador)
MIN_FREQ = 1
DIM = 32
EPOCAS = 30
LR = 0.01
VENTANA = 1
BATCH = 512
ruta_modelo = RUTA_MODELOS / f'word2vec_6g_mf{MIN_FREQ}.pt'

if ruta_modelo.exists():
    print(f'Cargando modelo cacheado: {ruta_modelo.name}')
    ck = torch.load(ruta_modelo, map_location=DISPOSITIVO, weights_only=False)
    idx_por_token = ck['idx_por_token']
    DIM = ck['dim']
    MIN_FREQ = ck['min_freq']
    modelo = ModeloWord2Vec(len(idx_por_token), DIM).to(DISPOSITIVO)
    modelo.load_state_dict(ck['state_dict'])
    modelo.eval()
    hist_principal = ck.get('historial', [])
else:
    print(f'Entrenando W2V con min_freq={MIN_FREQ}…')
    np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
    modelo, idx_por_token, hist_principal = entrenar_w2v(
        corpus, MIN_FREQ, DIM, EPOCAS, LR, VENTANA, BATCH, DISPOSITIVO)
    modelo.eval()
    torch.save({'state_dict': modelo.state_dict(),
                'idx_por_token': idx_por_token, 'dim': DIM,
                'min_freq': MIN_FREQ, 'historial': hist_principal}, ruta_modelo)
    print(f'→ guardado: {ruta_modelo.name}')

print(f'Modelo: vocab={len(idx_por_token):,}, dim={DIM}, min_freq={MIN_FREQ}')

Cargando modelo cacheado: word2vec_6g_mf1.pt


Modelo: vocab=2,817, dim=32, min_freq=1


## 3. Generación de trayectorias del deslizador

In [4]:
def generar_trayectoria(inicio: int, destino: int, n: int = 6):
    '''Migra token a token desde (inicio)*n hasta (destino)*n.'''
    tokens = [tuple([inicio]*n)]
    actual = [inicio]*n
    for _ in range(n):
        actual.pop(0)
        actual.append(destino)
        tokens.append(tuple(actual))
    return tokens

trayectoria_rem = generar_trayectoria(2, 4)
trayectoria_sws = generar_trayectoria(2, 3)

print('S2 → REM:')
for i, t in enumerate(trayectoria_rem):
    print(f'  paso {i}: {t}')
print('\nS2 → SWS:')
for i, t in enumerate(trayectoria_sws):
    print(f'  paso {i}: {t}')

S2 → REM:
  paso 0: (2, 2, 2, 2, 2, 2)
  paso 1: (2, 2, 2, 2, 2, 4)
  paso 2: (2, 2, 2, 2, 4, 4)
  paso 3: (2, 2, 2, 4, 4, 4)
  paso 4: (2, 2, 4, 4, 4, 4)
  paso 5: (2, 4, 4, 4, 4, 4)
  paso 6: (4, 4, 4, 4, 4, 4)

S2 → SWS:
  paso 0: (2, 2, 2, 2, 2, 2)
  paso 1: (2, 2, 2, 2, 2, 3)
  paso 2: (2, 2, 2, 2, 3, 3)
  paso 3: (2, 2, 2, 3, 3, 3)
  paso 4: (2, 2, 3, 3, 3, 3)
  paso 5: (2, 3, 3, 3, 3, 3)
  paso 6: (3, 3, 3, 3, 3, 3)


## 4. Auditoría: presencia de los tokens del deslizador en el corpus real

Los 6-gramas intermedios deberían ser **raros** (las transiciones en el sueño real son rápidas). Como aquí entrenamos con **min_freq=1**, incluso esos tokens poco frecuentes permanecen en el vocabulario y el deslizador queda bien cubierto.

In [5]:
frecuencias = Counter(corpus)
filas_audit = []
for ruta, traj in [('S2→REM', trayectoria_rem), ('S2→SWS', trayectoria_sws)]:
    for paso, tup in enumerate(traj):
        s = token_a_str(tup)
        f = frecuencias.get(s, 0)
        en_vocab = s in idx_por_token
        filas_audit.append({
            'ruta': ruta, 'paso': paso, 'token': s,
            'frecuencia': f, 'en_vocab': en_vocab
        })
tabla_audit = pd.DataFrame(filas_audit)
print(tabla_audit.to_string(index=False))
tabla_audit.to_csv(RUTA_STATS / 'deslizador_tokens_auditoria.csv', index=False)
print(f'\n→ guardado: deslizador_tokens_auditoria.csv')
print(f'\nCobertura: {tabla_audit["en_vocab"].sum()}/{len(tabla_audit)} tokens presentes en vocab')

  ruta  paso       token  frecuencia  en_vocab
S2→REM     0 2-2-2-2-2-2       37436      True
S2→REM     1 2-2-2-2-2-4         420      True
S2→REM     2 2-2-2-2-4-4         423      True
S2→REM     3 2-2-2-4-4-4         464      True
S2→REM     4 2-2-4-4-4-4         484      True
S2→REM     5 2-4-4-4-4-4         535      True
S2→REM     6 4-4-4-4-4-4       17620      True
S2→SWS     0 2-2-2-2-2-2       37436      True
S2→SWS     1 2-2-2-2-2-3         930      True
S2→SWS     2 2-2-2-2-3-3         436      True
S2→SWS     3 2-2-2-3-3-3         358      True
S2→SWS     4 2-2-3-3-3-3         396      True
S2→SWS     5 2-3-3-3-3-3         568      True
S2→SWS     6 3-3-3-3-3-3       12161      True

→ guardado: deslizador_tokens_auditoria.csv

Cobertura: 14/14 tokens presentes en vocab


## 5. Función de similitud sobre embeddings

In [6]:
def obtener_embedding(token_str):
    if token_str not in idx_por_token:
        return None
    idx = torch.tensor([idx_por_token[token_str]], device=DISPOSITIVO)
    with torch.no_grad():
        v = modelo.embeddings(idx).cpu().numpy()[0]
    return v

def similitud_coseno(a, b):
    if a is None or b is None:
        return None
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    if na == 0 or nb == 0:
        return None
    return float(np.dot(a, b) / (na * nb))

## 6. Similitud con el origen S2: bifurcación de trayectorias

Mide qué tan rápido se aleja cada trayectoria del estado S2 puro. **Si la H3 es cierta**, las dos curvas deben divergir antes del paso 3 (cuando el destino aún es minoría en la ventana).

In [7]:
def similitudes_con_origen(trayectoria):
    tokens_str = [token_a_str(t) for t in trayectoria]
    v0 = obtener_embedding(tokens_str[0])
    return [similitud_coseno(v0, obtener_embedding(t)) for t in tokens_str]

sim_origen_rem = similitudes_con_origen(trayectoria_rem)
sim_origen_sws = similitudes_con_origen(trayectoria_sws)

print('Paso  |  S2→REM  |  S2→SWS  |  Δ(SWS−REM)')
for i in range(len(trayectoria_rem)):
    a = sim_origen_rem[i]; b = sim_origen_sws[i]
    d = (b - a) if (a is not None and b is not None) else None
    print(f'  {i}   |  {a:+.4f}  |  {b:+.4f}  |  {d:+.4f}')

Paso  |  S2→REM  |  S2→SWS  |  Δ(SWS−REM)
  0   |  +1.0000  |  +1.0000  |  +0.0000
  1   |  +0.5970  |  +0.5676  |  -0.0294
  2   |  +0.4521  |  +0.4693  |  +0.0172
  3   |  +0.2459  |  +0.1810  |  -0.0649
  4   |  +0.2181  |  +0.1256  |  -0.0925
  5   |  +0.3648  |  +0.0349  |  -0.3299
  6   |  +0.3918  |  +0.1347  |  -0.2570


In [8]:
# Similitud ENTRE trayectorias (REM vs SWS) para subplot inferior
sim_entre_trayectorias = []
for tr, ts in zip(trayectoria_rem, trayectoria_sws):
    sim_entre_trayectorias.append(
        similitud_coseno(obtener_embedding(token_a_str(tr)),
                         obtener_embedding(token_a_str(ts)))
    )

COLOR_BAR_POS = '#16A085'   # teal/verde azulado
COLOR_BAR_NEG = '#C0392B'   # rojo

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1.4]})
pasos = list(range(len(trayectoria_rem)))

ax1.plot(pasos, sim_origen_rem, marker='o', lw=2.5, color=COLOR_REM, label='S2 → REM', markersize=9)
ax1.plot(pasos, sim_origen_sws, marker='s', lw=2.5, color=COLOR_SWS, label='S2 → SWS', markersize=9)
for p, v in zip(pasos, sim_origen_rem):
    ax1.annotate(f'{v:+.2f}', (p, v), textcoords='offset points', xytext=(0, 12),
                 ha='center', fontsize=9, color=COLOR_REM, fontweight='bold')
for p, v in zip(pasos, sim_origen_sws):
    ax1.annotate(f'{v:+.2f}', (p, v), textcoords='offset points', xytext=(0, -16),
                 ha='center', fontsize=9, color=COLOR_SWS, fontweight='bold')
ax1.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.6)
ax1.axvline(3, color='gray', lw=0.7, ls=':', alpha=0.5)
ax1.text(3.05, 1.12, 'punto medio\n(50% destino)', fontsize=8, color='gray', va='top')
ax1.set_ylabel('Similitud coseno con S2 puro')
ax1.set_title('Trayectorias en el espacio latente: alejamiento desde S2')
ax1.set_ylim(-1.25, 1.25)
ax1.grid(alpha=0.3)
ax1.legend(loc='upper right', framealpha=0.95)

# Panel inferior: BARRAS color teal/rojo
colores_barras = [COLOR_BAR_POS if v >= 0 else COLOR_BAR_NEG for v in sim_entre_trayectorias]
ax2.bar(pasos, sim_entre_trayectorias, color=colores_barras, edgecolor='black',
        linewidth=0.7, alpha=0.88, label='sim(REM_t, SWS_t)')
for p, v in zip(pasos, sim_entre_trayectorias):
    offset = 6 if v >= 0 else -14
    ax2.annotate(f'{v:+.3f}', (p, v), textcoords='offset points', xytext=(0, offset),
                 ha='center', fontsize=9, fontweight='bold')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('Paso del deslizador (épocas del destino en la ventana)')
ax2.set_ylabel('Similitud REM ↔ SWS')
ax2.set_xticks(pasos)
ax2.set_ylim(-1.25, 1.25)
ax2.grid(alpha=0.3, axis='y')
ax2.legend(loc='upper right', framealpha=0.95, fontsize=9)

plt.tight_layout()
plt.savefig(RUTA_IMG / 'similitud_origen.png', dpi=300, facecolor='white')
plt.close(fig)
print(f'PNG guardado: {RUTA_IMG / "similitud_origen.png"}')
print('\nSimilitud entre trayectorias REM↔SWS por paso:')
for p, (tr, ts, sim) in enumerate(zip(trayectoria_rem, trayectoria_sws, sim_entre_trayectorias)):
    print(f'  paso {p}: {tr}  vs  {ts}  →  sim = {sim:+.4f}')


PNG guardado: imagenes/deslizador/similitud_origen.png

Similitud entre trayectorias REM↔SWS por paso:
  paso 0: (2, 2, 2, 2, 2, 2)  vs  (2, 2, 2, 2, 2, 2)  →  sim = +1.0000
  paso 1: (2, 2, 2, 2, 2, 4)  vs  (2, 2, 2, 2, 2, 3)  →  sim = +0.5199
  paso 2: (2, 2, 2, 2, 4, 4)  vs  (2, 2, 2, 2, 3, 3)  →  sim = +0.2320
  paso 3: (2, 2, 2, 4, 4, 4)  vs  (2, 2, 2, 3, 3, 3)  →  sim = +0.2299
  paso 4: (2, 2, 4, 4, 4, 4)  vs  (2, 2, 3, 3, 3, 3)  →  sim = +0.2174
  paso 5: (2, 4, 4, 4, 4, 4)  vs  (2, 3, 3, 3, 3, 3)  →  sim = +0.1811
  paso 6: (4, 4, 4, 4, 4, 4)  vs  (3, 3, 3, 3, 3, 3)  →  sim = +0.1628


In [9]:
# Versión interactiva (Plotly): scatter arriba, BARRAS (teal) abajo
from plotly.subplots import make_subplots

COLOR_BAR_POS = '#16A085'
COLOR_BAR_NEG = '#C0392B'

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                    subplot_titles=('Similitud con S2 puro',
                                    'Similitud ENTRE trayectorias (REM ↔ SWS)'),
                    row_heights=[0.65, 0.35])

fig.add_trace(go.Scatter(x=pasos, y=sim_origen_rem, mode='lines+markers+text',
                         name='S2 → REM', line=dict(color=COLOR_REM, width=3),
                         marker=dict(size=11),
                         text=[f'{v:+.2f}' for v in sim_origen_rem],
                         textposition='top center',
                         textfont=dict(color=COLOR_REM, size=11),
                         customdata=[token_a_str(t) for t in trayectoria_rem],
                         hovertemplate='%{customdata}<br>sim=%{y:.4f}<extra></extra>'),
              row=1, col=1)
fig.add_trace(go.Scatter(x=pasos, y=sim_origen_sws, mode='lines+markers+text',
                         name='S2 → SWS', line=dict(color=COLOR_SWS, width=3),
                         marker=dict(size=11, symbol='square'),
                         text=[f'{v:+.2f}' for v in sim_origen_sws],
                         textposition='bottom center',
                         textfont=dict(color=COLOR_SWS, size=11),
                         customdata=[token_a_str(t) for t in trayectoria_sws],
                         hovertemplate='%{customdata}<br>sim=%{y:.4f}<extra></extra>'),
              row=1, col=1)
fig.add_hline(y=0, line=dict(color='gray', dash='dash', width=1), row=1, col=1)

colores_barras = [COLOR_BAR_POS if v >= 0 else COLOR_BAR_NEG for v in sim_entre_trayectorias]
fig.add_trace(go.Bar(x=pasos, y=sim_entre_trayectorias,
                     name='sim(REM, SWS)',
                     marker=dict(color=colores_barras, line=dict(color='black', width=0.6)),
                     text=[f'{v:+.3f}' for v in sim_entre_trayectorias],
                     textposition='outside',
                     textfont=dict(size=11),
                     cliponaxis=False,
                     hovertemplate='paso %{x}<br>sim=%{y:.4f}<extra></extra>'),
              row=2, col=1)
fig.add_hline(y=0, line=dict(color='black', width=1), row=2, col=1)

fig.update_xaxes(title_text='Paso del deslizador', row=2, col=1, tickmode='array', tickvals=pasos)
fig.update_yaxes(title_text='Similitud con S2', row=1, col=1, range=[-1.25, 1.25])
fig.update_yaxes(title_text='Similitud REM↔SWS', row=2, col=1, range=[-1.25, 1.25])
fig.update_layout(
    title='Deslizador: trayectorias y bifurcación',
    template='plotly_white', height=820, width=950,
    paper_bgcolor='white', plot_bgcolor='white', showlegend=True,
    margin=dict(t=80, b=60, l=70, r=40))
fig.write_html(RUTA_IMG / 'similitud_origen.html')
fig.show()


## 7. Control: modelo *surrogate* con corpus permutado

Se permuta el orden de las épocas (manteniendo la distribución marginal por paciente) y se reentrena un W2V idéntico. Si las trayectorias observadas dependen del orden temporal real, el modelo surrogate **debe** producir curvas sin separación.

**Caché:** si existe **word2vec_6g_surrogate_mf1.pt**, se carga en vez de reentrenar.

In [10]:
def construir_corpus_surrogate(df, n=6, semilla=42):
    rng = np.random.default_rng(semilla)
    corpus_s = []
    for pid, g in df.groupby('paciente', sort=False):
        seq = g['fase_num'].astype(int).tolist()
        rng.shuffle(seq)
        for i in range(len(seq) - n + 1):
            corpus_s.append(token_a_str(seq[i:i+n]))
    return corpus_s

ruta_surr = RUTA_MODELOS / f'word2vec_6g_surrogate_mf{MIN_FREQ}.pt'

if ruta_surr.exists():
    print(f'Cargando modelo surrogate cacheado: {ruta_surr.name}')
    ck_s = torch.load(ruta_surr, map_location=DISPOSITIVO, weights_only=False)
    idx_por_token_s = ck_s['idx_por_token']
    modelo_surr = ModeloWord2Vec(len(idx_por_token_s), ck_s['dim']).to(DISPOSITIVO)
    modelo_surr.load_state_dict(ck_s['state_dict'])
    modelo_surr.eval()
    hist_surr = ck_s.get('historial', [])
    print(f'  vocab={len(idx_por_token_s):,}, dim={ck_s["dim"]}, min_freq={ck_s["min_freq"]}')
else:
    print(f'Entrenando modelo surrogate (min_freq={MIN_FREQ})…')
    np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
    corpus_s = construir_corpus_surrogate(df, n=N, semilla=SEMILLA)
    print(f'  corpus surrogate: {len(corpus_s):,} ngramas')
    modelo_surr, idx_por_token_s, hist_surr = entrenar_w2v(
        corpus_s, MIN_FREQ, DIM, EPOCAS, LR, VENTANA, BATCH, DISPOSITIVO)
    modelo_surr.eval()
    torch.save({'state_dict': modelo_surr.state_dict(),
                'idx_por_token': idx_por_token_s, 'dim': DIM,
                'min_freq': MIN_FREQ, 'historial': hist_surr}, ruta_surr)
    print(f'→ guardado: {ruta_surr.name}')

Cargando modelo surrogate cacheado: word2vec_6g_surrogate_mf1.pt
  vocab=16,483, dim=32, min_freq=1


## 8. Trayectorias sobre el modelo surrogate

In [11]:
def obtener_embedding_surr(token_str):
    if token_str not in idx_por_token_s:
        return None
    idx = torch.tensor([idx_por_token_s[token_str]], device=DISPOSITIVO)
    with torch.no_grad():
        return modelo_surr.embeddings(idx).cpu().numpy()[0]

def similitudes_con_origen_modelo(trayectoria, get_emb):
    tokens_str = [token_a_str(t) for t in trayectoria]
    v0 = get_emb(tokens_str[0])
    return [similitud_coseno(v0, get_emb(t)) for t in tokens_str]

sim_origen_rem_surr = similitudes_con_origen_modelo(trayectoria_rem, obtener_embedding_surr)
sim_origen_sws_surr = similitudes_con_origen_modelo(trayectoria_sws, obtener_embedding_surr)

print('Paso  |  REM real  |  REM surr  |  SWS real  |  SWS surr')
for i in range(len(trayectoria_rem)):
    r = sim_origen_rem[i]; rs = sim_origen_rem_surr[i]
    s = sim_origen_sws[i]; ss = sim_origen_sws_surr[i]
    fmt = lambda x: f'{x:+.4f}' if x is not None else '   N/A '
    print(f'  {i}   |  {fmt(r)}  |  {fmt(rs)}  |  {fmt(s)}  |  {fmt(ss)}')

Paso  |  REM real  |  REM surr  |  SWS real  |  SWS surr
  0   |  +1.0000  |  +1.0000  |  +1.0000  |  +1.0000
  1   |  +0.5970  |  +0.6949  |  +0.5676  |  +0.6865
  2   |  +0.4521  |  +0.4674  |  +0.4693  |  +0.4258
  3   |  +0.2459  |  +0.0282  |  +0.1810  |  +0.1793
  4   |  +0.2181  |  +0.0769  |  +0.1256  |  +0.1167
  5   |  +0.3648  |  +0.1157  |  +0.0349  |  -0.0828
  6   |  +0.3918  |  +0.2026  |  +0.1347  |  +0.0565


## 8.5 Prueba de permutación: distribución nula con 50 subrogadas

La comparación contra **una sola** realización subrogada no constituye una prueba de significancia (su valor depende de la permutación concreta). Por eso construimos una **distribución nula** con **50 subrogadas** independientes (cada una con su propia permutación intra-paciente y su reentrenamiento completo de Word2Vec) y situamos la separación real frente a ella mediante un **p-valor empírico**. La figura **real_vs_surrogate.png** muestra la trayectoria real superpuesta a las **50 trayectorias subrogadas** (cada una como una línea tenue), y **null_distribution.png** el histograma nulo con el p-valor.

In [12]:
# === Prueba de permutación: distribución nula con 50 subrogadas ===
# Entrena 50 modelos W2V subrogados (orden temporal barajado intra-paciente, semillas 2000+k)
# y, para cada uno, mide (a) la separación acumulada REM-vs-SWS (escalar -> distribución nula)
# y (b) la trayectoria de similitud-con-origen por paso (-> 50 líneas). El modelo REAL es el cacheado.
# RESUMIBLE: cada subrogada se guarda al terminar; si se interrumpe, al re-ejecutar continúa.
import time

_NSURR = 50
_CKPT = RUTA_STATS / 'deslizador_surrogates_trayectorias.csv'

def _entrenar_surr(corpus_tokens, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    freq = Counter(corpus_tokens)
    vocab = sorted([t for t, c in freq.items() if c >= MIN_FREQ])
    idx = {t: i for i, t in enumerate(vocab)}
    indices = [idx[t] for t in corpus_tokens]
    loader = DataLoader(DatasetSkipGram(indices, VENTANA), batch_size=BATCH,
                        shuffle=True, num_workers=0)
    m = ModeloWord2Vec(len(vocab), DIM).to(DISPOSITIVO)
    crit = nn.CrossEntropyLoss(); opt = torch.optim.Adam(m.parameters(), lr=LR)
    for _ in range(EPOCAS):
        m.train()
        for a, b in loader:
            a, b = a.to(DISPOSITIVO), b.to(DISPOSITIVO)
            opt.zero_grad(); crit(m(a), b).backward(); opt.step()
    m.eval(); return m, idx

def _sims_modelo(m, idx, trayectoria):
    def emb(tok):
        if tok not in idx: return None
        i = torch.tensor([idx[tok]], device=DISPOSITIVO)
        with torch.no_grad(): return m.embeddings(i).cpu().numpy()[0]
    tk = [token_a_str(t) for t in trayectoria]; v0 = emb(tk[0])
    return [similitud_coseno(v0, emb(t)) for t in tk]

def _sep_traj(a, b):
    return float(np.nansum(np.abs(np.asarray(a, float) - np.asarray(b, float))))

# Estadístico observado: separación del modelo real (cacheado)
_sep_real = _sep_traj(sim_origen_rem, sim_origen_sws)

# Reanudar desde checkpoint si existe
_done = {}
if _CKPT.exists():
    _prev = pd.read_csv(_CKPT)
    for kk, sub in _prev.groupby('k'):
        d = {(row.ruta, int(row.paso)): row.sim for row in sub.itertuples()}
        _done[int(kk)] = {ru: [d.get((ru, p), np.nan) for p in range(N + 1)] for ru in ('REM', 'SWS')}
    print(f'Checkpoint: {len(_done)}/{_NSURR} subrogadas ya entrenadas (se reanuda).')

# 50 subrogadas (reentrenamiento independiente, semillas 2000+k)
_t0 = time.time()
_seps, _surr_traj = [], {'REM': [], 'SWS': []}
for k in range(_NSURR):
    if k in _done:
        sr, ss = _done[k]['REM'], _done[k]['SWS']
    else:
        cs = construir_corpus_surrogate(df, n=N, semilla=2000 + k)
        m_s, idx_s = _entrenar_surr(cs, 2000 + k)
        sr = [np.nan if v is None else v for v in _sims_modelo(m_s, idx_s, trayectoria_rem)]
        ss = [np.nan if v is None else v for v in _sims_modelo(m_s, idx_s, trayectoria_sws)]
        del m_s
        if DISPOSITIVO.type == 'cuda': torch.cuda.empty_cache()
        _rows = ([{'k': k, 'ruta': 'REM', 'paso': p, 'sim': sr[p]} for p in range(N + 1)] +
                 [{'k': k, 'ruta': 'SWS', 'paso': p, 'sim': ss[p]} for p in range(N + 1)])
        pd.DataFrame(_rows).to_csv(_CKPT, mode='a', header=not _CKPT.exists(), index=False)
        print(f'  subrogada {k + 1}/{_NSURR} entrenada (acumulado {time.time() - _t0:.0f}s)')
    _surr_traj['REM'].append(sr); _surr_traj['SWS'].append(ss)
    _seps.append(_sep_traj(sr, ss))
_seps = np.array(_seps)
_surr_traj = {r: np.array(_surr_traj[r], dtype=float) for r in ('REM', 'SWS')}

# p-valor empírico (una cola)
_mayores = int((_seps >= _sep_real).sum())
_pval = (1 + _mayores) / (1 + len(_seps))
_pct = 100 * (_seps < _sep_real).mean()
print(f'\nsep_real={_sep_real:.4f} | nulos: media={_seps.mean():.3f} sd={_seps.std():.3f} '
      f'| >= real: {_mayores}/{len(_seps)} | p={_pval:.4f} | percentil {_pct:.1f}')
pd.DataFrame({'k': range(len(_seps)), 'sep_surr': _seps, 'sep_real': _sep_real}
             ).to_csv(RUTA_STATS / 'deslizador_surrogates_null.csv', index=False)

# --- Figura 1: distribución nula (50 subrogadas) ---
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.hist(_seps, bins=30, color='#A6C9E2', edgecolor='#3498DB', alpha=0.85, label=f'Nulos (N={len(_seps)})')
ax.axvline(_sep_real, color='#C62828', lw=2.5, label=f'sep real = {_sep_real:.3f}')
ax.axvline(np.percentile(_seps, 95), color='gray', ls='--', lw=1.2, label='p95 nulo')
ax.set_xlabel('Separacion acumulada de trayectorias'); ax.set_ylabel('Frecuencia')
ax.set_title(f'Distribucion nula del deslizador (N={len(_seps)}) — p={_pval:.4f}, percentil {_pct:.1f}')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout()
plt.savefig(RUTA_IMG / 'null_distribution.png', dpi=180, facecolor='white'); plt.close(fig)
print('Figura: null_distribution.png')

# --- Figura 2: real vs las 50 subrogadas (líneas individuales, estilo LSTM) ---
fig, ax = plt.subplots(figsize=(10, 6))
for j in range(_NSURR):
    ax.plot(pasos, _surr_traj['REM'][j], '-', color=COLOR_SURR_REM, lw=0.6, alpha=0.30,
            label=f'S2→REM ({_NSURR} subrogadas)' if j == 0 else None)
for j in range(_NSURR):
    ax.plot(pasos, _surr_traj['SWS'][j], '-', color=COLOR_SURR_SWS, lw=0.6, alpha=0.30,
            label=f'S2→SWS ({_NSURR} subrogadas)' if j == 0 else None)
ax.plot(pasos, sim_origen_rem, 'o-', color=COLOR_REM, lw=2.8, markersize=9, label='S2→REM (real)')
ax.plot(pasos, sim_origen_sws, 's-', color=COLOR_SWS, lw=2.8, markersize=9, label='S2→SWS (real)')
ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.6)
ax.set_xlabel('Paso del deslizador'); ax.set_ylabel('Similitud coseno con S2 puro')
ax.set_title(f'Real vs. {_NSURR} subrogadas  (sep real={_sep_real:.3f}, p={_pval:.2f}, percentil {_pct:.0f})')
ax.set_xticks(pasos); ax.grid(alpha=0.3); ax.legend(loc='upper right', fontsize=9, framealpha=0.95)
plt.tight_layout()
plt.savefig(RUTA_IMG / 'real_vs_surrogate.png', dpi=300, facecolor='white'); plt.close(fig)
print('Figura: real_vs_surrogate.png (real vs 50 subrogadas, lineas)')


Checkpoint: 50/50 subrogadas ya entrenadas (se reanuda).

sep_real=0.7909 | nulos: media=0.698 sd=0.309 | >= real: 15/50 | p=0.3137 | percentil 70.0


Figura: null_distribution.png


Figura: real_vs_surrogate.png (real vs 50 subrogadas, lineas)


## 8.6 Robustez al stride: ventanas disjuntas

El corpus del deslizador usa solapamiento máximo (stride 1): dos 6-gramas consecutivos comparten cinco de sus seis épocas, lo que introduce autocorrelación entre tokens vecinos. Para medir cuánto de la separación de trayectorias depende de ese solapamiento, reentrenamos Word2Vec con **ventanas disjuntas (stride 6)** y con un paso intermedio (stride 3), y reconstruimos las trayectorias real vs surrogate.

In [13]:
# === Robustez al stride: ventanas DISJUNTAS (stride 6) y stride 3 ===
# Reentrena W2V con ventaneo disjunto/parcial y re-corre el deslizador real vs surrogate, para
# comprobar cuánto de la separación de trayectorias depende del solapamiento máximo (stride=1).
_df_stride = df[df['fase_num'] != 5].copy()   # descarta Sin_clasificar

def _corpus_stride(dframe, n, stride, shuffle=False, semilla=42):
    rng = np.random.default_rng(semilla); out = []
    for _, g in dframe.groupby('paciente', sort=False):
        seq = g['fase_num'].astype(int).tolist()
        if shuffle: rng.shuffle(seq)
        for i in range(0, len(seq) - n + 1, stride):
            out.append(token_a_str(seq[i:i + n]))
    return out

def _entrenar_stride(corpus_tokens, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    freq = Counter(corpus_tokens); vocab = sorted([t for t, c in freq.items() if c >= MIN_FREQ])
    idx = {t: i for i, t in enumerate(vocab)}
    indices = [idx[t] for t in corpus_tokens]
    loader = DataLoader(DatasetSkipGram(indices, VENTANA), batch_size=BATCH, shuffle=True, num_workers=0)
    m = ModeloWord2Vec(len(vocab), DIM).to(DISPOSITIVO)
    crit = nn.CrossEntropyLoss(); opt = torch.optim.Adam(m.parameters(), lr=LR)
    for _ in range(EPOCAS):
        m.train()
        for a, b in loader:
            a, b = a.to(DISPOSITIVO), b.to(DISPOSITIVO)
            opt.zero_grad(); crit(m(a), b).backward(); opt.step()
    m.eval(); return m, idx, len(vocab)

def _sims_st(m, idx, trayectoria):
    def emb(tok):
        if tok not in idx: return None
        i = torch.tensor([idx[tok]], device=DISPOSITIVO)
        with torch.no_grad(): return m.embeddings(i).cpu().numpy()[0]
    tk = [token_a_str(t) for t in trayectoria]; v0 = emb(tk[0])
    return [similitud_coseno(v0, emb(t)) for t in tk]

def _sep_st(a, b):
    return float(np.nansum([abs(x - y) for x, y in zip(a, b) if (x is not None and y is not None)]))

_resumen_stride = []
for stride in (6, 3):
    np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
    cr = _corpus_stride(_df_stride, N, stride, shuffle=False)
    cs = _corpus_stride(_df_stride, N, stride, shuffle=True, semilla=SEMILLA)
    m_r, idx_r, v_r = _entrenar_stride(cr, SEMILLA)
    m_s, idx_s, v_s = _entrenar_stride(cs, SEMILLA)
    srem = _sims_st(m_r, idx_r, trayectoria_rem); ssws = _sims_st(m_r, idx_r, trayectoria_sws)
    srem_s = _sims_st(m_s, idx_s, trayectoria_rem); ssws_s = _sims_st(m_s, idx_s, trayectoria_sws)
    sep_r = _sep_st(srem, ssws); sep_s = _sep_st(srem_s, ssws_s)
    razon = sep_r / max(sep_s, 1e-9)
    print(f'stride={stride}: sep_real={sep_r:.4f}  sep_surr={sep_s:.4f}  razon={razon:.3f}  vocab_real={v_r}')
    _resumen_stride.append({'stride': stride, 'vocab_real': v_r,
                            'sep_real': round(sep_r, 4), 'sep_surr': round(sep_s, 4), 'razon': round(razon, 3)})
    _f = lambda v: [x if x is not None else np.nan for x in v]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(pasos, _f(srem), 'o-', color=COLOR_REM, lw=2.5, label='S2→REM (real)')
    ax.plot(pasos, _f(ssws), 's-', color=COLOR_SWS, lw=2.5, label='S2→SWS (real)')
    ax.plot(pasos, _f(srem_s), 'o--', color=COLOR_SURR_REM, lw=2, label='S2→REM (surr)')
    ax.plot(pasos, _f(ssws_s), 's--', color=COLOR_SURR_SWS, lw=2, label='S2→SWS (surr)')
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_xlabel('Paso del deslizador'); ax.set_ylabel('Similitud coseno con S2 puro')
    ax.set_title(f'Deslizador con stride={stride} (real vs surrogate) — sep_real={sep_r:.2f}, razon={razon:.2f}')
    ax.set_xticks(pasos); ax.legend(fontsize=9); ax.grid(alpha=0.3); plt.tight_layout()
    plt.savefig(RUTA_IMG / f'stride{stride}_real_vs_surrogate.png', dpi=180, facecolor='white'); plt.close(fig)
    del m_r, m_s
    if DISPOSITIVO.type == 'cuda': torch.cuda.empty_cache()

pd.DataFrame(_resumen_stride).to_csv(RUTA_STATS / 'deslizador_stride_resumen.csv', index=False)
print('\nResumen stride:'); print(pd.DataFrame(_resumen_stride).to_string(index=False))
print('Figuras: stride6_real_vs_surrogate.png, stride3_real_vs_surrogate.png')

stride=6: sep_real=0.2338  sep_surr=0.9609  razon=0.243  vocab_real=781


stride=3: sep_real=0.6106  sep_surr=1.2618  razon=0.484  vocab_real=1035



Resumen stride:
 stride  vocab_real  sep_real  sep_surr  razon
      6         781    0.2338    0.9609  0.243
      3        1035    0.6106    1.2618  0.484
Figuras: stride6_real_vs_surrogate.png, stride3_real_vs_surrogate.png


## 9. Métricas cuantitativas

- **Separación**: |sim_REM(t) − sim_SWS(t)| acumulada en pasos 1..6 (más separación = más bifurcación).
- **AUC**: área bajo la curva de similitud con origen (más baja = se aleja más rápido).
- **Razón real/surrogate** de separación: > 1 indica que la estructura observada depende del orden temporal.

In [14]:
def auc(y):
    y2 = [v if v is not None else 0 for v in y]
    return float(np.trapz(y2, dx=1))

def separacion_acumulada(a, b):
    return float(sum(abs(x-y) for x, y in zip(a, b) if (x is not None and y is not None)))

auc_rem = auc(sim_origen_rem)
auc_sws = auc(sim_origen_sws)
auc_rem_s = auc(sim_origen_rem_surr)
auc_sws_s = auc(sim_origen_sws_surr)
sep_real = separacion_acumulada(sim_origen_rem, sim_origen_sws)
sep_surr = separacion_acumulada(sim_origen_rem_surr, sim_origen_sws_surr)
razon = sep_real / max(sep_surr, 1e-9)

metricas = pd.DataFrame([{
    'auc_origen_rem_real': auc_rem,
    'auc_origen_sws_real': auc_sws,
    'auc_origen_rem_surr': auc_rem_s,
    'auc_origen_sws_surr': auc_sws_s,
    'separacion_acumulada_real': sep_real,
    'separacion_acumulada_surr': sep_surr,
    'razon_real_sobre_surr': razon,
}])
metricas.to_csv(RUTA_STATS / 'deslizador_metricas.csv', index=False)
print(metricas.T.to_string(header=False))
print(f'\n→ guardado: deslizador_metricas.csv')

auc_origen_rem_real        2.573921
auc_origen_sws_real        1.945844
auc_origen_rem_surr        1.984371
auc_origen_sws_surr        1.853773
separacion_acumulada_real  0.790935
separacion_acumulada_surr  0.585483
razon_real_sobre_surr      1.350911

→ guardado: deslizador_metricas.csv


/tmp/ipykernel_213962/4286757742.py:3: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.



In [15]:
# Tabla detallada por paso (real + surrogate de una realización ilustrativa)
filas_t = []
for ruta, traj, real, surr in [
    ('S2→REM', trayectoria_rem, sim_origen_rem, sim_origen_rem_surr),
    ('S2→SWS', trayectoria_sws, sim_origen_sws, sim_origen_sws_surr),
]:
    for p, tup in enumerate(traj):
        filas_t.append({
            'ruta': ruta, 'paso': p, 'token': token_a_str(tup),
            'sim_origen_real': real[p],
            'sim_origen_surrogate': surr[p],
        })
tabla_t = pd.DataFrame(filas_t)
tabla_t.to_csv(RUTA_STATS / 'deslizador_trayectorias.csv', index=False)
print(tabla_t.to_string(index=False))
print('\n→ guardado: deslizador_trayectorias.csv')

  ruta  paso       token  sim_origen_real  sim_origen_surrogate
S2→REM     0 2-2-2-2-2-2         1.000000              1.000000
S2→REM     1 2-2-2-2-2-4         0.597028              0.694947
S2→REM     2 2-2-2-2-4-4         0.452094              0.467412
S2→REM     3 2-2-2-4-4-4         0.245943              0.028205
S2→REM     4 2-2-4-4-4-4         0.218130              0.076858
S2→REM     5 2-4-4-4-4-4         0.364848              0.115668
S2→REM     6 4-4-4-4-4-4         0.391755              0.202563
S2→SWS     0 2-2-2-2-2-2         1.000000              1.000000
S2→SWS     1 2-2-2-2-2-3         0.567636              0.686510
S2→SWS     2 2-2-2-2-3-3         0.469261              0.425813
S2→SWS     3 2-2-2-3-3-3         0.181012              0.179332
S2→SWS     4 2-2-3-3-3-3         0.125647              0.116666
S2→SWS     5 2-3-3-3-3-3         0.034936             -0.082817
S2→SWS     6 3-3-3-3-3-3         0.134705              0.056536

→ guardado: deslizador_trayectorias.csv

## 10. Conclusiones del Deslizador (interpretación de las gráficas)

### 10.1 Trayectorias hacia REM y SWS (gráfica **similitud_origen.png**)

- **Paso 0** (**2-2-2-2-2-2**, S2 puro): ambas trayectorias parten de similitud +1.00 con el origen — son el mismo punto por construcción.
- **Pasos 1–2**: caída suave y casi paralela (REM +0.60 → +0.45, SWS +0.57 → +0.47). El espacio latente "no decide" todavía: las dos trayectorias se mueven juntas alejándose de S2.
- **Paso 3** (50 % destino, **2-2-2-X-X-X**): **bifurcación visible**. REM cae a +0.25 y SWS a +0.18; comienza la separación.
- **Pasos 4–6**: REM se **estabiliza** entre +0.22 y +0.39 (rebote ligero), mientras SWS sigue descendiendo hasta +0.03 en el paso 5 y vuelve a +0.13 en el paso 6.
- **Interpretación geométrica**: en el espacio latente Word2Vec, el camino **hacia SWS implica un alejamiento más profundo desde S2** que el camino hacia REM. Esto es coherente con la neurofisiología: REM mantiene cierta similitud con S2 (ambas presentan actividad EEG rápida y de bajo voltaje), mientras SWS rompe radicalmente con esa configuración (predominio de ondas lentas δ).

### 10.2 Bifurcación REM ↔ SWS (panel inferior con barras)

- Paso 0: similitud REM↔SWS = +1.000 (mismo token).
- Paso 1: +0.520 — todavía vecinos en el espacio latente.
- Paso 2 en adelante: caída monotónica (+0.232, +0.230, +0.217, +0.181, +0.163).
- **Las trayectorias nunca se vuelven ortogonales** (mínimo +0.163), pero la similitud entre ellas **decae sostenidamente paso a paso**, confirmando que cada inserción de una época destino las empuja a vecindarios cada vez más distintos.

### 10.3 Prueba de permutación: 50 subrogadas (**real_vs_surrogate.png**, **null_distribution.png**)

Una **única** realización subrogada produce una separación menor que la real (razón ~1.35), lo que *a primera vista* sugeriría que el orden temporal aporta la diferencia. Pero una sola permutación no es una prueba: su valor depende de la realización concreta.

Al construir la **distribución nula con 50 subrogadas** (cada una con su permutación intra-paciente y su reentrenamiento completo de Word2Vec), la separación real **no se distingue del azar**: cae dentro del haz de las 50 subrogadas y su **p-valor empírico es ~0.3** (queda en torno al percentil 70 del nulo). El motivo es que, al permutar, el vocabulario de 6-gramas se dispara (de unos 2800 a unos 16000 tokens) y los *embeddings* de las trayectorias sintéticas adquieren mucha varianza, de modo que con frecuencia separan *más* que los datos reales por puro azar.

**Conclusión honesta:** el deslizador **valida la geometría del espacio de embeddings** (las rutas hacia REM y SWS son direcciones distintas y reproducibles), pero **no constituye una prueba de anticipación**: su separación no es estadísticamente significativa frente al nulo. La evidencia propiamente dicha de anticipación sobre datos reales la aporta la **LSTM** (notebook 10).

### 10.4 Robustez al stride (**stride6_real_vs_surrogate.png**, **stride3_real_vs_surrogate.png**)

Con **ventanas disjuntas (stride 6)** —que eliminan la autocorrelación entre tokens vecinos a costa de un corpus seis veces menor— la separación de trayectorias se **reduce drásticamente**, lo que confirma que buena parte de la estructura observada con stride 1 proviene del solapamiento de ventanas y no de una anticipación temporal genuina. Es coherente con el resultado de la prueba de permutación.

### 10.5 Resumen ejecutivo

1. **Geometría organizada**: el embedding de cada paso intermedio del deslizador carga información del destino; REM y SWS ocupan direcciones distintas y separables.
2. **REM y SWS no son simétricas**: SWS se aleja más de S2; REM mantiene parentesco con la fase de origen.
3. **No es prueba de anticipación**: la separación **no es significativa** frente a la distribución nula de 50 subrogadas (p~0.3) y **colapsa** con ventanas disjuntas. El deslizador es **ilustrativo del espacio latente**, no una demostración de anticipación.
4. **La evidencia de anticipación sobre datos reales** descansa en el modelo recurrente **LSTM** (notebook 10), que sí supera su propio control con secuencias barajadas.